# Pandas DataFrames: The Essentials

Inspect, select, filter, sort, and modify tabular data with pandas.

> 🧭 This notebook focuses on practical data work. Examples are intentionally compact and do not cover machine learning.

## Learning goals
- Understand Series and DataFrame objects.
- Inspect columns, types, missingness, and summary statistics.
- Select rows and columns with `loc` and `iloc`.
- Filter and sort records without confusing labels and positions.

In [1]:
import pandas as pd

df = pd.DataFrame({
    "city": ["Baku", "Ganja", "Baku", "Shaki", "Ganja"],
    "year": [2023, 2023, 2024, 2024, 2024],
    "visits": [120, 75, 138, 54, 91],
    "rating": [4.5, 4.1, 4.6, None, 4.2],
})
display(df.head())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Dtypes:\n", df.dtypes)
df.info()

Shape: (5, 4)
Columns: ['city', 'year', 'visits', 'rating']
Dtypes:
 city       object
year        int64
visits      int64
rating    float64
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   city    5 non-null      object 
 1   year    5 non-null      int64  
 2   visits  5 non-null      int64  
 3   rating  4 non-null      float64
dtypes: float64(1), int64(2), object(1)
memory usage: 292.0+ bytes


,city,year,visits,rating
0,Baku,2023,120,4.5
1,Ganja,2023,75,4.1
2,Baku,2024,138,4.6
3,Shaki,2024,54,NaN
4,Ganja,2024,91,4.2


## Selection and filtering
- `df["column"]` returns a Series.
- `df.loc[label, columns]` selects by index labels.
- `df.iloc[position, columns]` selects by integer position.
- Use `&` and `|` with parentheses for multiple pandas conditions; Python's `and` / `or` do not work element-wise.

In [2]:
print("A column:\n", df["visits"])
display(df.loc[1:3, ["city", "visits"]])
display(df.iloc[0:2, 0:3])

mask = (df["year"] == 2024) & (df["visits"] >= 90)
display(df.loc[mask])

A column:
 0    120
1     75
2    138
3     54
4     91
Name: visits, dtype: int64


,city,visits
1,Ganja,75
2,Baku,138
3,Shaki,54


,city,year,visits
0,Baku,2023,120
1,Ganja,2023,75


,city,year,visits,rating
2,Baku,2024,138,4.6
4,Ganja,2024,91,4.2


## Sorting, derived columns, and safe assignment
Use `.assign()` to make transformation steps easy to chain. Avoid chained assignment such as `df[df.x > 0]["y"] = ...`; use `.loc` instead.

In [3]:
sorted_df = df.sort_values(["year", "visits"], ascending=[False, False])
display(sorted_df)

df = df.assign(is_high_traffic=lambda x: x["visits"] >= 100)
df.loc[df["rating"].isna(), "rating"] = 0  # Demonstration only; zero is not always a valid imputation.
display(df)

,city,year,visits,rating
2,Baku,2024,138,4.6
4,Ganja,2024,91,4.2
3,Shaki,2024,54,NaN
0,Baku,2023,120,4.5
1,Ganja,2023,75,4.1


,city,year,visits,rating,is_high_traffic
0,Baku,2023,120,4.5,True
1,Ganja,2023,75,4.1,False
2,Baku,2024,138,4.6,True
3,Shaki,2024,54,0.0,False
4,Ganja,2024,91,4.2,False


## Summary methods and common surprises
`count()` excludes missing values; `size` counts rows. `describe()` defaults to numeric columns unless `include="all"` is used. An index is not automatically a data column.

In [4]:
print("Numeric summary:")
display(df.describe())
print("All-column summary:")
display(df.describe(include="all"))
print("Non-missing ratings:", df["rating"].count())
print("Number of rows:", len(df))
print("Missing values by column:\n", df.isna().sum())

Numeric summary:
All-column summary:
Non-missing ratings: 5
Number of rows: 5
Missing values by column:
 city               0
year               0
visits             0
rating             0
is_high_traffic    0
dtype: int64


,year,visits,rating
count,5.000000,5.000000,5.000000
mean,2023.600000,95.600000,3.480000
std,0.547723,33.797929,1.956272
min,2023.000000,54.000000,0.000000
25%,2023.000000,75.000000,4.100000
50%,2024.000000,91.000000,4.200000
75%,2024.000000,120.000000,4.500000
max,2024.000000,138.000000,4.600000


,city,year,visits,rating,is_high_traffic
count,5,5.000000,5.000000,5.000000,5
unique,3,NaN,NaN,NaN,2
top,Baku,NaN,NaN,NaN,False
freq,2,NaN,NaN,NaN,3
mean,NaN,2023.600000,95.600000,3.480000,NaN
std,NaN,0.547723,33.797929,1.956272,NaN
min,NaN,2023.000000,54.000000,0.000000,NaN
25%,NaN,2023.000000,75.000000,4.100000,NaN
50%,NaN,2024.000000,91.000000,4.200000,NaN
75%,NaN,2024.000000,120.000000,4.500000,NaN


## Practice
Select 2024 records with more than 100 visits, sort them from highest to lowest visits, and display only city and visits.

In [5]:
answer = df.loc[(df["year"] == 2024) & (df["visits"] > 100), ["city", "visits"]].sort_values("visits", ascending=False)
display(answer)

,city,visits
2,Baku,138
